# 01 - Explore the Premier League data

Step 1 of the project: load the Transfermarkt tables and check data quality for
Premier League seasons 2022/23-2025/26 before building anything.

Run `python3 scripts/download_data.py` first.

Notes:
- The dataset labels seasons by their **start year**: `2025` means 2025/26.
- Valuation rule: a season's value is the **latest valuation on or before June 30** after it ends.

In [1]:
import pandas as pd

RAW = "../data/raw"
SEASONS = [2022, 2023, 2024, 2025]
LEAGUE = "GB1"  # Premier League

players = pd.read_csv(f"{RAW}/players.csv.gz")
valuations = pd.read_csv(f"{RAW}/player_valuations.csv.gz", parse_dates=["date"])
appearances = pd.read_csv(f"{RAW}/appearances.csv.gz", parse_dates=["date"])
games = pd.read_csv(f"{RAW}/games.csv.gz")

## Premier League appearances

`appearances` has no season column, so we attach it from `games`.
Each appearance is one player in one match.

In [2]:
pl_games = games[(games.competition_id == LEAGUE) & games.season.isin(SEASONS)][["game_id", "season"]]
pl_apps = appearances.merge(pl_games, on="game_id")
pl_apps.groupby("season").agg(games=("game_id", "nunique"), appearances=("appearance_id", "size"))

,games,appearances
season,,
2022,380,11345
2023,380,11384
2024,380,11567
2025,380,11492


## Players per season

Players with at least one Premier League appearance. Squad players who never
played are not in `appearances`, so they don't show up here.

In [3]:
player_seasons = (
    pl_apps.groupby(["player_id", "season"])
    .agg(apps=("game_id", "size"), minutes=("minutes_played", "sum"),
         goals=("goals", "sum"), assists=("assists", "sum"))
    .reset_index()
)
player_seasons.groupby("season").agg(
    players=("player_id", "size"),
    median_minutes=("minutes", "median"),
    under_450_min=("minutes", lambda m: (m < 450).sum()),  # fewer than ~5 full games
)

,players,median_minutes,under_450_min
season,,,
2022,554,1224.5,146
2023,570,1208.5,166
2024,562,1157.0,165
2025,537,1328.0,137


## Valuation coverage at the June 30 cutoff

For each player-season, find the latest valuation on or before June 30 after the
season ends (e.g. 2025/26 -> 2026-06-30), and how old that valuation is.
Upstream valuations stop at 2026-06-12, so 2025/26 relies on values up to that date.

In [4]:
player_seasons["cutoff"] = pd.to_datetime((player_seasons.season + 1).astype(str) + "-06-30")

# merge_asof finds, for each row, the latest valuation at or before its cutoff date
matched = pd.merge_asof(
    player_seasons.sort_values("cutoff"),
    valuations[["player_id", "date", "market_value_in_eur"]].sort_values("date"),
    left_on="cutoff", right_on="date", by="player_id", direction="backward",
).rename(columns={"date": "valuation_date", "market_value_in_eur": "value_eur"})
matched["valuation_age_days"] = (matched.cutoff - matched.valuation_date).dt.days

matched.groupby("season").agg(
    player_seasons=("player_id", "size"),
    has_value=("value_eur", lambda v: v.notna().mean().round(3)),
    median_age_days=("valuation_age_days", "median"),
    older_than_180_days=("valuation_age_days", lambda d: (d > 180).sum()),
)

,player_seasons,has_value,median_age_days,older_than_180_days
season,,,,
2022,554,0.978,10.0,16
2023,570,0.996,34.0,26
2024,562,0.986,31.0,11
2025,537,0.976,27.0,13


Rows with **no** valuation at all, and the stalest ones. These are pilot candidates.

In [5]:
names = players[["player_id", "name", "position", "date_of_birth"]]
no_value = matched[matched.value_eur.isna()].merge(names, on="player_id")
print(f"{len(no_value)} player-seasons with no valuation")
no_value[["name", "season", "apps", "minutes"]].head(10)

35 player-seasons with no valuation


,name,season,apps,minutes
0,Owen Bevan,2022,1,8
1,Matthew Craig,2022,1,13
2,Shea Charles,2022,1,27
3,Kami Doyle,2022,1,6
4,David Ozoh,2022,1,1
5,Jack Hinshelwood,2022,1,1
6,Ethan Nwaneri,2022,1,1
7,George Abbott,2022,1,1
8,Lewis Miley,2022,1,14
9,Samuel Amo-Ameyaw,2022,1,3


In [6]:
stale = matched[matched.valuation_age_days > 180].merge(names, on="player_id")
stale.sort_values("valuation_age_days", ascending=False)[
    ["name", "season", "valuation_date", "valuation_age_days", "minutes"]
].head(10)

,name,season,valuation_date,valuation_age_days,minutes
63,Patrick Roberts,2025,2023-06-01,1125.0,27
61,George Earthy,2025,2024-05-27,764.0,8
49,Alex Palmer,2024,2023-06-01,760.0,1170
48,Wes Burns,2024,2023-06-09,752.0,934
45,Jakub Stolarczyk,2024,2023-10-27,612.0,900
47,Christian Walton,2024,2023-10-27,612.0,630
52,Patson Daka,2024,2023-10-27,612.0,721
50,Alfie Dorrington,2024,2023-12-19,559.0,13
42,Paris Maghoma,2024,2023-12-27,551.0,112
43,Ben Mee,2024,2024-05-27,399.0,181


## Missing values in key columns

In [7]:
pl_players = players[players.player_id.isin(player_seasons.player_id)]
pd.DataFrame({
    "players: missing": pl_players[["date_of_birth", "position", "sub_position"]].isna().sum(),
}).T

,date_of_birth,position,sub_position
players: missing,0,0,0


In [8]:
pl_apps[["goals", "assists", "minutes_played"]].isna().sum().to_frame("appearances: missing").T

,goals,assists,minutes_played
appearances: missing,0,0,0


## Duplicates

In [9]:
pd.Series({
    "duplicate appearance_id": pl_apps.appearance_id.duplicated().sum(),
    "same player twice in one game": pl_apps.duplicated(["player_id", "game_id"]).sum(),
    "duplicate valuation (player, date)": valuations.duplicated(["player_id", "date"]).sum(),
    "players with >1 club in a season": pl_apps.groupby(["player_id", "season"]).player_club_id.nunique().gt(1).sum(),
})

duplicate appearance_id                0
same player twice in one game          0
duplicate valuation (player, date)     0
players with >1 club in a season      51
dtype: int64